In [1]:
# ============================================================
# CELL 1: SETUP AND IMPORTS
# ============================================================
import pandas as pd
import numpy as np
import joblib
import time
import gc
from pathlib import Path

LOCAL_DIR = Path("C:\\Users\\admin\\Documents\\Glacier Project")
REGIONS = ['R1a', 'R1b', 'R2', 'R3']
PIXEL_AREA_M2 = 100  # 10m × 10m pixel = 100 m²

EASD_FEATURES = ['elevation', 'aspect', 'slope', 'edge_distance']
N_PCS = 10
PC_FEATURES = [f'PC{i+1}' for i in range(N_PCS)]
ALL_FEATURES = EASD_FEATURES + PC_FEATURES

print('Setup complete.')

Setup complete.


In [2]:
# ============================================================
# CELL 2: LOAD DATA AND TRAINED MODELS
# 
# Loads the full 15.2M-pixel Peru-wide dataset and the trained 
# RF models from the previous notebook. The PCA + scaler are
# applied here to get PC scores attached to each pixel.
# ============================================================
t0 = time.time()

dfs = []
for region in REGIONS:
    df = pd.read_parquet(LOCAL_DIR / f'Merged\\{region.lower()}_combined.parquet')
    df['region'] = region
    dfs.append(df)
df_all = pd.concat(dfs, ignore_index=True)
print(f'Loaded {len(df_all):,} pixels in {time.time()-t0:.1f}s')

# Load PCA artifacts
pca = joblib.load(LOCAL_DIR / 'Saved_Models\\pca_alphaearth.joblib')
scaler = joblib.load(LOCAL_DIR / 'Saved_Models\\scaler_alphaearth.joblib')

# Load trained RF models
rf_easd = joblib.load(LOCAL_DIR / 'Saved_Models\\rf_easd_new_baseline.joblib')
rf_pc10 = joblib.load(LOCAL_DIR / 'Saved_Models\\rf_easd_pc10.joblib')
print('Loaded PCA, scaler, and both RF models.')

Loaded 15,242,639 pixels in 33.5s
Loaded PCA, scaler, and both RF models.


In [3]:
# ============================================================
# CELL 3: PROJECT ALL PIXELS TO PC SPACE
# 
# Applies the saved StandardScaler + PCA to attach PC1-PC10 scores
# to every pixel in the dataset. Memory-aware: deletes intermediate
# arrays after use.
# ============================================================
ae_cols = [c for c in df_all.columns 
           if c.startswith('A') and len(c) == 3 and c[1:].isdigit()]
assert len(ae_cols) == 64, 'Wrong number of AE bands'

t0 = time.time()
X_ae = df_all[ae_cols].values.astype(np.float32)
X_scaled = scaler.transform(X_ae)
del X_ae; gc.collect()

X_pca = pca.transform(X_scaled)[:, :N_PCS]
del X_scaled; gc.collect()

df_all[PC_FEATURES] = X_pca
del X_pca; gc.collect()

print(f'Attached PC1-PC{N_PCS} scores to all {len(df_all):,} pixels '
      f'in {time.time()-t0:.1f}s')
print(f'Memory: {df_all.memory_usage(deep=True).sum() / 1e9:.2f} GB')

Attached PC1-PC10 scores to all 15,242,639 pixels in 48.8s
Memory: 5.91 GB


In [4]:
# ============================================================
# CELL 4: PREDICT MELT PROBABILITIES FOR ALL PIXELS
# 
# Runs both the baseline (EASD-only) and AlphaEarth (EASD+PC10) 
# models on every pixel to get per-pixel melt probability.
# Returns the probability of class 1 (melt) for each pixel.
# 
# This is the slowest cell - probably 1-3 minutes per model on 15M
# pixels. Both predictions stored for downstream comparison.
# ============================================================
t0 = time.time()

X_easd = df_all[EASD_FEATURES].values
df_all['prob_easd'] = rf_easd.predict_proba(X_easd)[:, 1]
print(f'EASD-only predictions: {time.time()-t0:.1f}s')
del X_easd; gc.collect()

t1 = time.time()
X_pc10 = df_all[ALL_FEATURES].values
df_all['prob_pc10'] = rf_pc10.predict_proba(X_pc10)[:, 1]
print(f'EASD+PC10 predictions: {time.time()-t1:.1f}s')
del X_pc10; gc.collect()

# Quick sanity check on probabilities
print(f'\nProbability distributions:')
print(f'  EASD-only:  mean={df_all["prob_easd"].mean():.3f}, '
      f'median={df_all["prob_easd"].median():.3f}')
print(f'  EASD+PC10:  mean={df_all["prob_pc10"].mean():.3f}, '
      f'median={df_all["prob_pc10"].median():.3f}')
print(f'  Actual melt rate: {df_all["melt_label"].mean():.3f}')

EASD-only predictions: 203.1s
EASD+PC10 predictions: 183.0s

Probability distributions:
  EASD-only:  mean=0.369, median=0.317
  EASD+PC10:  mean=0.325, median=0.240
  Actual melt rate: 0.195


In [5]:
# ============================================================
# CELL 5: COMPUTE OBSERVED TOTAL MELT AREA (THE THRESHOLDING TARGET)
#
# The thresholded prediction works as follows:
#   1. Count actual melt pixels and convert to area (km²).
#   2. Rank all pixels by predicted melt probability (descending).
#   3. Take the top-N pixels whose total area equals observed melt.
#   4. Those top-N pixels are the model's "predicted melt set".
#   5. Compare to actual melt set via overlap metrics.
# 
# This step computes step 1.
# ============================================================
n_actual_melt = (df_all['melt_label'] == 1).sum()
n_total = len(df_all)

actual_melt_area_km2 = n_actual_melt * PIXEL_AREA_M2 / 1e6
total_ice_area_km2 = n_total * PIXEL_AREA_M2 / 1e6

print(f'=== Population stats ===')
print(f'  Total ice pixels:    {n_total:,}')
print(f'  Actual melt pixels:  {n_actual_melt:,}')
print(f'  Melt rate:           {n_actual_melt/n_total*100:.2f}%')
print(f'  Total ice area:      {total_ice_area_km2:.1f} km²')
print(f'  Actual melt area:    {actual_melt_area_km2:.1f} km²')
print(f'\n(For reference: Darina\'s reported 2016-2023 melt was 217.8 km².)')

=== Population stats ===
  Total ice pixels:    15,242,639
  Actual melt pixels:  2,971,684
  Melt rate:           19.50%
  Total ice area:      1524.3 km²
  Actual melt area:    297.2 km²

(For reference: Darina's reported 2016-2023 melt was 217.8 km².)


In [6]:
# ============================================================
# CELL 6: THRESHOLD AND COMPUTE OVERLAP - HELPER FUNCTION
# 
# Generic function: given pixel probabilities and actual labels,
# threshold to match observed melt area, then compute several 
# overlap metrics:
#   - Overlap %: (predicted ∩ actual) / actual  (i.e. recall)
#   - IoU:        (predicted ∩ actual) / (predicted ∪ actual)
#   - Precision:  (predicted ∩ actual) / predicted
#   - Threshold value used (probability cutoff)
# ============================================================
def thresholded_overlap(probs, labels, target_n_melt):
    """
    Threshold predictions to predict exactly target_n_melt pixels as melt.
    Compare against actual labels.
    
    probs: array of predicted melt probabilities, shape (N,)
    labels: array of true binary labels (0 = non-melt, 1 = melt), shape (N,)
    target_n_melt: integer, number of pixels to predict as melt
                   (typically equals actual count of melt pixels)
    """
    # Get indices of top-N most-vulnerable pixels
    threshold_value = np.partition(probs, -target_n_melt)[-target_n_melt]
    predicted_melt = probs >= threshold_value
    
    # In case of ties at the threshold, the actual count may exceed target.
    # Trim using argsort to get exactly target_n_melt predicted melts.
    if predicted_melt.sum() != target_n_melt:
        top_indices = np.argsort(probs)[::-1][:target_n_melt]
        predicted_melt = np.zeros(len(probs), dtype=bool)
        predicted_melt[top_indices] = True
    
    actual_melt = labels == 1
    
    n_predicted = predicted_melt.sum()
    n_actual = actual_melt.sum()
    n_intersect = (predicted_melt & actual_melt).sum()
    n_union = (predicted_melt | actual_melt).sum()
    
    overlap_pct = n_intersect / n_actual * 100  # equivalent to recall
    precision_pct = n_intersect / n_predicted * 100
    iou = n_intersect / n_union
    
    return {
        'threshold': threshold_value,
        'n_predicted': n_predicted,
        'n_actual': n_actual,
        'n_intersect': n_intersect,
        'overlap_pct': overlap_pct,
        'precision_pct': precision_pct,
        'iou': iou,
    }

# Sanity check the function
print('Helper function defined.')

Helper function defined.


In [7]:
# ============================================================
# CELL 7: COMPUTE OVERLAP FOR BOTH MODELS
# 
# Runs the thresholding + overlap computation for the baseline and
# the AlphaEarth-augmented model. Reports the comparison.
# ============================================================
print('=== EASD-only model ===')
result_easd = thresholded_overlap(
    df_all['prob_easd'].values,
    df_all['melt_label'].values,
    target_n_melt=n_actual_melt
)
for k, v in result_easd.items():
    if isinstance(v, float):
        print(f'  {k:15s} {v:.4f}')
    else:
        print(f'  {k:15s} {v:,}')

print('\n=== EASD + PC1-PC10 model ===')
result_pc10 = thresholded_overlap(
    df_all['prob_pc10'].values,
    df_all['melt_label'].values,
    target_n_melt=n_actual_melt
)
for k, v in result_pc10.items():
    if isinstance(v, float):
        print(f'  {k:15s} {v:.4f}')
    else:
        print(f'  {k:15s} {v:,}')

# Comparison summary
print('\n=== COMPARISON ===')
print(f'{"Metric":<25}{"EASD-only":<15}{"EASD+PC10":<15}{"Δ":<10}')
print(f'{"Overlap % (recall)":<25}'
      f'{result_easd["overlap_pct"]:<15.2f}'
      f'{result_pc10["overlap_pct"]:<15.2f}'
      f'{result_pc10["overlap_pct"]-result_easd["overlap_pct"]:+.2f}')
print(f'{"Precision %":<25}'
      f'{result_easd["precision_pct"]:<15.2f}'
      f'{result_pc10["precision_pct"]:<15.2f}'
      f'{result_pc10["precision_pct"]-result_easd["precision_pct"]:+.2f}')
print(f'{"IoU":<25}'
      f'{result_easd["iou"]:<15.4f}'
      f'{result_pc10["iou"]:<15.4f}'
      f'{result_pc10["iou"]-result_easd["iou"]:+.4f}')

print(f'\n(Reference: Darina\'s reported 2016-2023 overlap was 74.9%)')

=== EASD-only model ===
  threshold       0.6500
  n_predicted     2,971,684
  n_actual        2,971,684
  n_intersect     1,617,252
  overlap_pct     54.4221
  precision_pct   54.4221
  iou             0.3738

=== EASD + PC1-PC10 model ===
  threshold       0.6167
  n_predicted     2,971,684
  n_actual        2,971,684
  n_intersect     2,023,781
  overlap_pct     68.1022
  precision_pct   68.1022
  iou             0.5163

=== COMPARISON ===
Metric                   EASD-only      EASD+PC10      Δ         
Overlap % (recall)       54.42          68.10          +13.68
Precision %              54.42          68.10          +13.68
IoU                      0.3738         0.5163         +0.1425

(Reference: Darina's reported 2016-2023 overlap was 74.9%)


In [8]:
# ============================================================
# CELL 8: SAVE RESULTS
# 
# Save the per-pixel predictions for downstream analysis
# (per-glacier overlap, visualisation, etc.). The full df with
# probabilities is large but we only need a subset of columns.
# ============================================================
output_cols = ['region', 'lon', 'lat', 'melt_label', 'edge_distance',
               'prob_easd', 'prob_pc10']
df_predictions = df_all[output_cols]
df_predictions.to_parquet(LOCAL_DIR / 'Results\\predictions_full_peru.parquet')
print(f'Saved predictions: {len(df_predictions):,} rows')
print(f'Columns: {df_predictions.columns.tolist()}')

Saved predictions: 15,242,639 rows
Columns: ['region', 'lon', 'lat', 'melt_label', 'edge_distance', 'prob_easd', 'prob_pc10']
